<h1 style='text-align: center; font-family: Colonna MT; font-size: 28px; font-weight: 600'>RISK-RETURN PERFORMANCE ANALYSIS OF  DSE EQUITIES<br> <span style='text-align: center; font-family: Bell MT; font-size: 20px; font-weight: 600'>Dar es Salaam Stock Exchange (DSE) — Professional Portfolio Analytics</span><br> <span style='text-align: center; font-family: Cursive; font-size: 12px; font-weight: 600'>Author: Chausiku Kassimu | WhatsApp: +255 790 708 450 | Email: chausikukassimu1@gmail</span></h1>

----



<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>1.0. IMPORT REQUIRED LIBRARIES</h1>

In [105]:
    # ── Core | Data Manipulation and Handling
    from typing import List, Optional, Union, Dict
    from datetime import datetime, timedelta
    from scr.simulated  import _simulation
    from scr.loader  import _loader
    from scr.config import TICKERS
    import pandas as pd
    import numpy as np
    import warnings
    import math
    import os
    
    # ── Data Visualizations
    from plotly.subplots import make_subplots
    import plotly.graph_objects as go
    import plotly.express as px
    
    import matplotlib.gridspec as gridspec
    import matplotlib.patches as mpatches
    import matplotlib.ticker as mticker
    import matplotlib.dates as mdates
    import matplotlib.ticker as mtick
    import matplotlib.pyplot as plt
    import networkx as nx
    import seaborn as sns
    
    # ── Scientific
    from scipy.stats import norm, jarque_bera, shapiro
    from sklearn.preprocessing import MinMaxScaler
    from scipy.stats import skew, kurtosis
    from scr.statistikis  import *
    from scipy import stats

<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>2.0. VISUAL CONFIGURATION SETTINGS</h1>

In [2]:
    # Warnings Suppression, Maximum columns, and Floating Point Display Format 
    pd.set_option('display.float_format', lambda x: '%.2f' % x)
    pd.set_option('display.max_columns', 100)
    warnings.simplefilter("ignore") 

    # ── Visual Configuration
    DARK        = dict(
                    paper_bgcolor="#0D1117",
                    plot_bgcolor="#0D1117",
                    font=dict(family="Inter, Arial, sans-serif", color="#C9D1D9", size=12),
                    xaxis=dict(gridcolor="#21262D", linecolor="#30363D", zerolinecolor="#30363D"),
                    yaxis=dict(gridcolor="#21262D", linecolor="#30363D", zerolinecolor="#30363D"),
                    margin=dict(l=60, r=40, t=60, b=50),
                    )
    
    ACCENT        = "#00C49F"
    BENCH_C       = "#D29922"
    RED           = "#F85149"
    COLORS        = ["#00C49F","#0088FE","#D29922","#FF6B6B","#A78BFA","#34D399","#F59E0B"]
    POS_COLOR     = '#2ecc71'
    NEG_COLOR     = '#e74c3c'
    TEXT_COLOR    = '#2c3e50'

    plt.rcParams.update({'font.family': 'Arial'})
    plt.rcParams.update({'font.size': 9 })
    sns.set_palette("tab20c")

<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>3.0. USEFULLY CONSTANTS</h1>

In [3]:
    TRADING_DAYS        = 252         # Annual trading days
    RISK_FREE_RATE      = 0.08        # 8% — Tanzania T-Bill proxy
    CONFIDENCE_LEVEL    = 0.95        # 95% VaR
    MC_SIMULATIONS      = 10_000      # Monte Carlo paths
    MC_HORIZON          = 252         # 1-year projection
    LOOK_BACK           = 710
    DATA_FOLDER         = './Datasets'

<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>4.0. IMPORTING AND PREPARE DATASETS</h1>

In [112]:
#df                    =  pd.read_excel("DSE.xlsx", sheet_name='data')
df                    =  _loader(DATA_FOLDER, LOOK_BACK)
#df                    =  _simulation(TICKERS, LOOK_BACK)

prices                =  (df.pivot_table(
                            index   = ['Date'],
                            columns = 'Stock',
                            values  = 'Close',
                            aggfunc = 'first')
                            .reset_index()
                             )
prices.columns        =  [f"{col}" if col is not None else "" for col in prices.columns]
prices                =  prices.set_index('Date')
prices['DSI']         =  prices.sum(axis=1)/prices.shape[1]-1
TICKERS               =  prices.columns

display(prices[prices.columns[-18:]].tail(20))

,AFRIPRISE,CRDB,DCB,DSE,MBP,MCB,MKCB,MUCOBA,NICO,PAL,SWISS,TBL,TCC,TCCL,TOL,TTP,VODA,DSI
Date,,,,,,,,,,,,,,,,,,
2026-02-18,855.00,3140.00,465.00,6760.00,2810.00,1170.00,5080.00,820.00,3830.00,340.00,2700.00,10310.00,12020.00,3180.00,885.00,370.00,795.00,3265.47
2026-02-19,845.00,3060.00,480.00,6780.00,2820.00,1320.00,5160.00,825.00,3830.00,315.00,2580.00,10300.00,12050.00,3200.00,885.00,400.00,795.00,3272.24
2026-02-20,850.00,2940.00,480.00,6850.00,2500.00,1420.00,4930.00,850.00,3750.00,315.00,2710.00,10270.00,12100.00,3200.00,890.00,395.00,795.00,3248.71
2026-02-23,855.00,2980.00,495.00,6810.00,2830.00,1520.00,4890.00,850.00,3790.00,310.00,2660.00,10230.00,12160.00,3180.00,890.00,395.00,820.00,3273.41
2026-02-26,855.00,3000.00,525.00,6770.00,2580.00,1660.00,5320.00,855.00,3790.00,315.00,2670.00,10190.00,12000.00,3180.00,915.00,390.00,805.00,3282.53
2026-02-27,850.00,3020.00,595.00,6720.00,2860.00,1900.00,5340.00,855.00,3850.00,330.00,2690.00,10130.00,12020.00,3140.00,1005.00,420.00,800.00,3324.00
2026-03-02,860.00,3010.00,660.00,6740.00,3100.00,2140.00,5100.00,855.00,3970.00,355.00,2670.00,9990.00,12000.00,3200.00,1010.00,480.00,820.00,3349.59
2026-03-03,860.00,2910.00,755.00,6420.00,2910.00,2260.00,5070.00,980.00,3880.00,400.00,2680.00,9960.00,11980.00,3240.00,1000.00,450.00,810.00,3326.35
2026-03-04,850.00,2790.00,835.00,6500.00,2660.00,2430.00,4980.00,1080.00,3820.00,450.00,2680.00,9820.00,11930.00,3180.00,1010.00,435.00,800.00,3307.82


<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>5.0. DATA OVERVIEW AND QUALITY CHECKS</h1>

<h3 style='font-family: Bradley Hand ITC; font-size: 18px; font-weight: 600'> 5.1: DATA SHAPE AND COLUMNS</h3>

In [113]:
print(f"{prices.shape[0]}: rows, {prices.shape[1]}: columns")

710: rows, 18: columns


In [114]:
prices.columns

Index(['AFRIPRISE', 'CRDB', 'DCB', 'DSE', 'MBP', 'MCB', 'MKCB', 'MUCOBA',
       'NICO', 'PAL', 'SWISS', 'TBL', 'TCC', 'TCCL', 'TOL', 'TTP', 'VODA',
       'DSI'],
      dtype='object')

In [115]:
prices.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 710 entries, 2023-05-04 to 2026-03-19
Data columns (total 18 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   AFRIPRISE  710 non-null    float64
 1   CRDB       710 non-null    float64
 2   DCB        710 non-null    float64
 3   DSE        710 non-null    float64
 4   MBP        710 non-null    float64
 5   MCB        710 non-null    float64
 6   MKCB       710 non-null    float64
 7   MUCOBA     710 non-null    float64
 8   NICO       710 non-null    float64
 9   PAL        710 non-null    float64
 10  SWISS      710 non-null    float64
 11  TBL        710 non-null    float64
 12  TCC        710 non-null    float64
 13  TCCL       710 non-null    float64
 14  TOL        710 non-null    float64
 15  TTP        710 non-null    float64
 16  VODA       710 non-null    float64
 17  DSI        710 non-null    float64
dtypes: float64(18)
memory usage: 105.4 KB


<h3 style='font-family: Bradley Hand ITC; font-size: 18px; font-weight: 600'> 5.2: COLUMNS SUMMARIES</h3>

In [8]:
def _audit(df: pd.DataFrame) -> pd.DataFrame:
    results = []
    for col_name in df.columns:
        distinct_values_counts    = df[col_name].value_counts()
        top_10_values_counts      = df[col_name].value_counts().head(10).to_dict()
        distinct_values_counts    = ({k: v for k, v in sorted(top_10_values_counts.items(), key=lambda 
            item: item[1], reverse=True)})
        results.append({
            'col_name':               col_name,
            'col_dtype':              df[col_name].dtype,
            'nulls':           df[col_name].isnull().sum(),
            'non_nulls':       df[col_name].notnull().sum(),
            'distinct_values': df[col_name].nunique(),
            "distinct_values_counts": distinct_values_counts
            })
    
    return pd.DataFrame(results)

results = _audit(prices)
display(results)

,col_name,col_dtype,nulls,non_nulls,distinct_values,distinct_values_counts
0,AFRIPRISE,float64,0,852,852,"{94.98004480442064: 1, 92.91693239815054: 1, 9..."
1,CRDB,float64,0,852,852,"{105.51836485030243: 1, 110.0305786533727: 1, ..."
2,DCB,float64,0,852,852,"{91.51519186179458: 1, 91.18370061502796: 1, 9..."
3,DSE,float64,0,852,852,"{98.90771920312108: 1, 101.8457863113954: 1, 9..."
4,EABL,float64,0,852,852,"{118.89395070054913: 1, 116.24266419217031: 1,..."
5,ITRUST ETF,float64,0,852,852,"{114.20674880352405: 1, 116.96608736901248: 1,..."
6,JATU,float64,0,852,852,"{86.28992225447445: 1, 88.95457125829496: 1, 8..."
7,JHL,float64,0,852,852,"{90.78098650429442: 1, 92.0115143127943: 1, 91..."
8,KA,float64,0,852,852,"{83.15741421078434: 1, 85.42915633028271: 1, 8..."
9,KCB,float64,0,852,852,"{113.02708844767653: 1, 121.72027370880743: 1,..."


<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>6.0. RETURNS COMPUTATIONS AND STATISTICS</h1>

<h2 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 18px; text-align: left'>6.2. STOCKS RETURNS COMPUTATIONS</h2>

In [116]:
returns  =  prices.pct_change().dropna()
display(returns[returns.columns[-20:]].tail(20))

,AFRIPRISE,CRDB,DCB,DSE,MBP,MCB,MKCB,MUCOBA,NICO,PAL,SWISS,TBL,TCC,TCCL,TOL,TTP,VODA,DSI
Date,,,,,,,,,,,,,,,,,,
2026-02-18,0.00,0.05,0.03,-0.01,0.12,0.15,0.05,0.09,0.02,0.10,0.00,0.01,0.00,0.01,-0.01,0.00,-0.02,0.02
2026-02-19,-0.01,-0.03,0.03,0.00,0.00,0.13,0.02,0.01,0.00,-0.07,-0.04,-0.00,0.00,0.01,0.00,0.08,0.00,0.00
2026-02-20,0.01,-0.04,0.00,0.01,-0.11,0.08,-0.04,0.03,-0.02,0.00,0.05,-0.00,0.00,0.00,0.01,-0.01,0.00,-0.01
2026-02-23,0.01,0.01,0.03,-0.01,0.13,0.07,-0.01,0.00,0.01,-0.02,-0.02,-0.00,0.00,-0.01,0.00,0.00,0.03,0.01
2026-02-26,0.00,0.01,0.06,-0.01,-0.09,0.09,0.09,0.01,0.00,0.02,0.00,-0.00,-0.01,0.00,0.03,-0.01,-0.02,0.00
2026-02-27,-0.01,0.01,0.13,-0.01,0.11,0.14,0.00,0.00,0.02,0.05,0.01,-0.01,0.00,-0.01,0.10,0.08,-0.01,0.01
2026-03-02,0.01,-0.00,0.11,0.00,0.08,0.13,-0.04,0.00,0.03,0.08,-0.01,-0.01,-0.00,0.02,0.00,0.14,0.02,0.01
2026-03-03,0.00,-0.03,0.14,-0.05,-0.06,0.06,-0.01,0.15,-0.02,0.13,0.00,-0.00,-0.00,0.01,-0.01,-0.06,-0.01,-0.01
2026-03-04,-0.01,-0.04,0.11,0.01,-0.09,0.08,-0.02,0.10,-0.02,0.12,0.00,-0.01,-0.00,-0.02,0.01,-0.03,-0.01,-0.01


<h2 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 18px; text-align: left'>6.3. STOCKS RETURNS STATISTICS</h2>

In [117]:
def _statistics(df: pd.DataFrame, tickers: Optional[List[str]] = None) -> pd.DataFrame:
    results = []
    for ticker in tickers:
        CAGR          =  ann_return(df[ticker])
        vol           =  ann_vol(df[ticker])
        sh            =  sharpe(df[ticker])
        so            =  sortino(df[ticker])
        cal           =  calmar(df[ticker])
        
        counts        =  df[ticker].count()
        mean_val      =  df[ticker].mean()
        median_val    =  df[ticker].median()
        mode_val      =  df[ticker].mode().iloc[0] if not df[ticker].mode().empty else np.nan
        std_dev       =  df[ticker].std()
        variance      =  df[ticker].var()
        value_range   =  df[ticker].max() - df[ticker].min()
        iqr           =  df[ticker].quantile(0.75) - df[ticker].quantile(0.25)
        
        skewness_val  =  skew(df[ticker], nan_policy='omit')
        kurtosis_val  =  kurtosis(df[ticker], nan_policy='omit')
        jb_stat, jb_p =  jarque_bera(df[ticker])
        jb_p_interp   =  'Non-Normal' if jb_p < 0.05 else 'Normal'
        jb_p_fmt      =  f'<0.05' if jb_p < 0.05 else f'{jb_p:.3f}'
        
        

        results.append({
            'Ticker':       ticker,
            'Counts':       counts,
            'Mean':         mean_val,
            'Median':       median_val,
            'Mode':         mode_val,
            'Std':          std_dev,
            'Variance':     variance,
            'Range':        value_range,
            'IQR':          iqr,
            'Skewness':     skewness_val,
            'Kurtosis':     kurtosis_val,
            'JB p-value':   jb_p_fmt,
            'JB Notes':     jb_p_interp,
            'CAGR':         CAGR,
            'Vol':          vol,
            'Sharpe':       sh,
            'Sortino':      so,
            'Calmer':       cal,
        })

    return pd.DataFrame(results)

if __name__ == "__main__":
    results    = _statistics(returns, returns.columns)
    display(results)

,Ticker,Counts,Mean,Median,Mode,Std,Variance,Range,IQR,Skewness,Kurtosis,JB p-value,JB Notes,CAGR,Vol,Sharpe,Sortino,Calmer
0,AFRIPRISE,709,0.00,0.00,0.00,0.02,0.00,0.37,0.00,1.79,16.12,<0.05,Non-Normal,0.83,0.39,1.52,1.88,2.95
1,CRDB,709,0.00,0.00,0.00,0.02,0.00,0.14,0.01,0.50,2.76,<0.05,Non-Normal,0.83,0.26,2.11,2.91,2.62
2,DCB,709,0.00,0.00,0.00,0.03,0.00,0.29,0.00,0.99,6.35,<0.05,Non-Normal,0.62,0.51,1.04,1.33,1.35
3,DSE,709,0.00,0.00,0.00,0.02,0.00,0.27,0.00,1.35,11.91,<0.05,Non-Normal,0.59,0.33,1.31,1.34,2.46
4,MBP,709,0.00,0.00,0.00,0.03,0.00,0.30,0.00,0.72,8.70,<0.05,Non-Normal,1.01,0.51,1.46,1.16,1.71
5,MCB,709,0.00,0.00,0.00,0.03,0.00,0.32,0.00,1.34,11.49,<0.05,Non-Normal,0.85,0.47,1.37,1.08,2.44
6,MKCB,709,0.00,0.00,0.00,0.03,0.00,0.30,0.00,1.45,8.92,<0.05,Non-Normal,0.91,0.49,1.41,1.42,2.44
7,MUCOBA,709,0.00,0.00,0.00,0.02,0.00,0.30,0.00,2.79,39.22,<0.05,Non-Normal,0.33,0.29,0.86,0.36,1.41
8,NICO,709,0.00,0.00,0.00,0.03,0.00,0.28,0.01,0.64,5.56,<0.05,Non-Normal,1.18,0.49,1.68,2.13,2.61
9,PAL,709,0.00,0.00,0.00,0.03,0.00,0.29,0.00,0.43,10.81,<0.05,Non-Normal,0.13,0.46,0.33,0.25,0.22


<h2 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 18px; text-align: left'>6.5. INDIVIDUAL STOCK STATISTICS | INDIVIDUAL STOCK RISK-RETURN PERFORMANCE STATISTICS</h2>

In [118]:
returns      = prices.pct_change().dropna()
stock_ret    = returns[TICKERS]
dsi_ret      = returns["DSI"]

def _statistiks(stock_ret:pd.DataFrame, TICKERS: List[str]):
    stock_stats = []
    for TICKER in TICKERS:
        r            = stock_ret[TICKER]
        ar           = ann_return(r)
        av           = ann_vol(r)
        sh           = sharpe(r)
        so           = sortino(r)
        
        mdd, td, dd_s, rec = max_dd(r)
        cal                = calmar(r)
        var95, cvar95      = var_cvar(r)
        om                 = omega_ratio(r)
        b, a, r2           = beta_alpha(r, dsi_ret)
        jb_stat, jb_p      = jarque_bera(r)
    
        stock_stats.append({
            "Ticker":          TICKER,
            "Ann Return (%)":  round(ar*100, 2),
            "Ann Vol (%)":     round(av*100, 2),
            "Sharpe":          round(sh, 3),
            "Sortino":         round(so, 3),
            "Calmar":          round(cal, 3),
            "Max DD (%)":      round(mdd*100, 2),
            "Recovery Days":   rec if rec else "In DD",
            "VaR 95% (%)":     round(var95*100, 3),
            "CVaR 95% (%)":    round(cvar95*100, 3),
            "Omega":           round(om, 3),
            "Beta":            round(b, 3),
            "Alpha (%)":       round(a*100, 2),
            "R²":              round(r2, 3),
            "JB p-value":      f"{jb_p:.3f}",
            "Skewness":        round(r.skew(), 3),
            "Excess Kurtosis": round(r.kurtosis(), 3),
        })

    return pd.DataFrame(stock_stats)#.set_index("Ticker")

statistiks = _statistiks(stock_ret, stock_ret.columns)
display(statistiks[statistiks.columns[:-3]])

,Ticker,Ann Return (%),Ann Vol (%),Sharpe,Sortino,Calmar,Max DD (%),Recovery Days,VaR 95% (%),CVaR 95% (%),Omega,Beta,Alpha (%),R²
0,AFRIPRISE,82.50,39.38,1.52,1.88,2.95,-28.00,307,-2.56,-4.50,1.50,0.53,76.41,0.01
1,CRDB,82.51,26.41,2.11,2.91,2.62,-31.45,111,-2.15,-3.60,1.61,1.03,64.22,0.11
2,DCB,62.13,51.27,1.04,1.32,1.35,-46.15,435,-3.97,-7.03,1.33,1.63,54.45,0.07
3,DSE,58.69,33.35,1.31,1.34,2.46,-23.90,34,-2.14,-4.28,1.49,1.49,41.59,0.14
4,MBP,100.94,51.13,1.46,1.16,1.71,-58.88,26,-2.83,-7.53,1.62,3.18,74.14,0.27
5,MCB,85.13,46.87,1.37,1.08,2.44,-34.86,9,-1.77,-6.51,1.69,0.70,82.76,0.02
6,MKCB,91.04,48.51,1.41,1.42,2.44,-37.27,In DD,-3.49,-6.54,1.58,2.71,67.92,0.22
7,MUCOBA,33.41,29.03,0.86,0.36,1.41,-23.73,In DD,0.00,-0.11,1.73,0.12,27.72,0.00
8,NICO,117.72,48.50,1.68,2.13,2.60,-45.19,42,-3.46,-6.57,1.47,2.09,99.14,0.13
9,PAL,13.41,46.42,0.33,0.25,0.22,-60.00,146,-3.29,-7.57,1.13,0.57,12.89,0.01


<h1 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 20px; text-align: left'>6.0. DAR ES SALAAM STOCK EXCHANGE (DSE) MARKERT PERFORMANCE</h1>

In [119]:
from scr.models import modeli, _return_volatility

In [131]:
def fomatter(df: pd.DataFrame, cols: list, div: float = 1, show_unit: bool = False) -> pd.DataFrame:
    df_out     =   df.copy()
    units      =   { 1: "", 1e3: "K", 1e6: "M", 1e9: "B"}
    unit       =   units.get(div, "")
    suffix     =   f" {unit}" if show_unit and unit else ""

    def _fmt(x):
        if pd.isna(x):
            return np.nan
        return f"{x / div:,.2f}{suffix}"

    valid_cols           =  [c for c in cols if c in df_out.columns]
    df_out[valid_cols]   =  (df_out[valid_cols].apply(lambda s: s.map(_fmt)))

    return df_out


    
cols           =  ['Today', 'PD', 'DoD', 'WTD', 'PW', 'WoW', 'MTD', 'PM', 'MoM', 'QTD', 'PQ', 'QoQ']
model          =  modeli(df, freqs=['W', 'M', "Q"])
#turnover       =  fomatter(model.compute("Turnover"), cols, 1e6)
volume         =  fomatter(model.compute("Volume"), cols, 1e3)

<h2 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 18px; text-align: left'>6.5. VOLUME PERFORMANCE SNAPSHOT</h2>

In [121]:
display(volume.style.background_gradient(cmap="RdYlGn"))

,Stock,Today,PD,DoD,DoD %,WTD,PW,WoW,WoW %,MTD,PM,MoM,MoM %,QTD,PQ,QoQ,QoQ %
0,AFRIPRISE,9.43,7.80,1.63,20.912938,17.23,44.97,-27.74,-61.689496,28.72,250.93,-222.21,-88.555465,28.72,727.42,-698.70,-96.052080
1,CRDB,15.69,18.14,-2.45,-13.500551,33.83,47.75,-13.92,-29.154189,51.81,215.74,-163.93,-75.986039,51.81,623.22,-571.41,-91.687218
2,DCB,8.82,14.79,-5.97,-40.381236,23.61,48.05,-24.44,-50.857404,46.27,198.30,-152.02,-76.664683,46.27,667.02,-620.75,-93.062716
3,DSE,12.58,8.98,3.60,40.031170,21.56,78.05,-56.49,-72.374827,53.17,263.05,-209.88,-79.785366,53.17,692.72,-639.54,-92.323694
4,EABL,16.93,6.16,10.77,175.016244,23.09,72.19,-49.10,-68.020944,49.75,254.26,-204.52,-80.434509,49.75,678.89,-629.14,-92.672145
5,ITRUST ETF,15.23,12.27,2.96,24.142147,27.50,37.47,-9.97,-26.607953,46.26,208.18,-161.92,-77.781140,46.26,640.25,-593.99,-92.775433
6,JATU,8.94,5.91,3.03,51.218274,14.85,39.88,-25.03,-62.770812,27.83,209.66,-181.83,-86.724697,27.83,712.75,-684.91,-96.094968
7,JHL,3.10,8.69,-5.59,-64.277497,11.80,46.45,-34.65,-74.601158,25.53,206.12,-180.59,-87.613406,25.53,642.60,-617.07,-96.026903
8,KA,2.26,3.37,-1.11,-32.947462,5.63,74.11,-68.48,-92.405576,32.92,257.85,-224.93,-87.232161,32.92,718.67,-685.75,-95.419032
9,KCB,18.15,19.88,-1.72,-8.673777,38.03,54.71,-16.68,-30.491683,63.69,210.33,-146.64,-69.718437,63.69,674.54,-610.85,-90.557954


<h2 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 18px; text-align: left'>6.5. TURNOVER PERFORMANCE SNAPSHOT</h2>

In [123]:
display(turnover.style.background_gradient(cmap="RdYlGn"))

,Stock,Today,PD,DoD,DoD %,WTD,PW,WoW,WoW %,MTD,PM,MoM,MoM %,QTD,PQ,QoQ,QoQ %
0,AFRIPRISE,0.90,0.74,0.16,21.234074,1.63,4.24,-2.60,-61.411802,2.71,23.84,-21.13,-88.623114,2.71,68.98,-66.27,-96.067734
1,CRDB,1.66,1.91,-0.26,-13.479911,3.57,5.15,-1.58,-30.722501,5.56,23.06,-17.50,-75.896026,5.56,67.15,-61.59,-91.722803
2,DCB,0.81,1.43,-0.62,-43.600477,2.24,4.54,-2.30,-50.649032,4.41,18.49,-14.08,-76.160815,4.41,62.01,-57.61,-92.893818
3,DSE,1.24,0.90,0.34,38.229390,2.14,7.72,-5.57,-72.213831,5.35,25.93,-20.59,-79.386780,5.35,68.63,-63.28,-92.211808
4,EABL,2.01,0.71,1.30,182.377185,2.73,8.39,-5.66,-67.495138,5.86,29.69,-23.83,-80.250389,5.86,79.48,-73.61,-92.621614
5,ITRUST ETF,1.74,1.39,0.35,24.889899,3.13,4.35,-1.22,-28.006006,5.29,24.00,-18.72,-77.978201,5.29,73.57,-68.28,-92.814793
6,JATU,0.77,0.51,0.26,51.551030,1.28,3.45,-2.17,-62.936132,2.41,18.10,-15.69,-86.678197,2.41,61.43,-59.01,-96.074836
7,JHL,0.28,0.79,-0.51,-64.420737,1.07,4.29,-3.22,-74.963161,2.35,19.02,-16.67,-87.658363,2.35,59.21,-56.86,-96.035291
8,KA,0.19,0.28,-0.10,-33.890028,0.47,6.32,-5.85,-92.530525,2.79,21.98,-19.19,-87.289773,2.79,61.45,-58.66,-95.453206
9,KCB,2.05,2.32,-0.27,-11.595622,4.37,6.36,-1.99,-31.253368,7.36,24.67,-17.31,-70.156730,7.36,79.33,-71.97,-90.719697


<h2 style='font-family: Bradley Hand ITC; font-weight: 600; font-size: 18px; text-align: left'>6.5. RISK-RETURNS PROFILE</h2>

In [132]:
features    = _return_volatility(df, annualize=True)
_return     =  (features.sort_values(["Stock", "Date"])
              .reset_index(drop=True)
              .groupby("Stock")
              .tail(1)
              )

display(_return)

,Date,Stock,Close,1 D,1W Return,1M Return,3M Return,6M Return,1Y Return,1W Vol,1M Vol,3M Vol,6M Vol,1Y Vol
709,2026-03-19,AFRIPRISE,815.00,0.00,-1.81,-5.23,69.79,68.04,270.45,17.61,13.27,80.95,61.44,50.15
1419,2026-03-19,CRDB,2880.00,0.00,-4.64,0.00,118.18,148.28,284.00,21.42,52.72,48.66,40.45,36.19
2129,2026-03-19,DCB,740.00,0.00,-9.76,68.18,214.89,184.62,492.00,34.50,116.91,106.77,87.51,72.78
2839,2026-03-19,DSE,6600.00,0.00,-1.05,-1.93,-5.85,6.45,184.48,8.85,24.49,34.49,43.36,47.85
3549,2026-03-19,MBP,2600.00,0.00,-4.76,14.54,312.70,195.45,664.71,22.31,120.97,133.46,104.42,81.49
4259,2026-03-19,MCB,1810.00,0.00,9.04,90.53,297.80,325.88,483.87,127.65,141.53,115.65,99.07,77.94
4969,2026-03-19,MKCB,4820.00,0.00,-3.60,-6.59,120.09,105.11,731.03,40.54,70.86,94.59,88.66,77.00
5679,2026-03-19,MUCOBA,900.00,0.00,-17.43,19.21,119.51,164.71,125.00,97.54,94.27,86.64,64.29,48.53
6389,2026-03-19,NICO,3660.00,0.00,-0.81,-1.61,140.79,136.13,394.59,4.90,38.43,81.27,64.96,68.54
7099,2026-03-19,PAL,570.00,0.00,-35.23,65.22,159.09,200.00,42.50,92.52,151.80,110.39,100.56,77.89
